# 05 — Final Comparison: NAM vs XGBoost

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml

with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

from src.data.split import load_splits
from src.models.nam_trainer import NAMTrainer
from src.models.xgboost_baseline import train_xgboost, extract_shap_values
from src.conformal.wrapper import NAMSklearnWrapper, create_conformal_classifier, predict_with_confidence
from src.conformal.calibration import compute_all_calibration_metrics, conformal_coverage_analysis
from src.evaluation.metrics import compute_all_metrics, bootstrap_metrics
from src.evaluation.statistical_tests import delong_test, mcnemar_test
from src.evaluation.comparison import create_comparison_table
from src.visualization.shape_functions import get_feature_importance_from_nam
from src.visualization.shap_plots import plot_nam_vs_shap_comparison
from src.visualization.paper_figures import plot_roc_curves, plot_feature_importance_comparison

## 1. Load Data & Train Both Models

In [ ]:
splits = load_splits(config['data']['processed_dir'])
X_train = splits['X_train'].values
X_cal = splits['X_cal'].values
X_test = splits['X_test'].values
y_train, y_cal, y_test = splits['y_train'], splits['y_cal'], splits['y_test']
feature_names = splits['feature_names']

# NAM
trainer = NAMTrainer(device='cpu')
nam_model, nam_hist = trainer.train_model(X_train, y_train, X_cal, y_cal, config['nam'])
nam_model.eval()
with torch.no_grad():
    logits, _ = nam_model(torch.FloatTensor(X_test))
    y_prob_nam = torch.sigmoid(logits).numpy().ravel()

# XGBoost
xgb_model = train_xgboost(X_train, y_train, X_cal, y_cal, config['xgboost'])
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(f'NAM best val AUC: {nam_hist["best_val_auc"]:.4f}')

## 2. Point Metrics

In [ ]:
nam_metrics = compute_all_metrics(y_test, y_prob_nam)
xgb_metrics = compute_all_metrics(y_test, y_prob_xgb)

print('NAM Metrics:')
for k, v in nam_metrics.items():
    print(f'  {k:>15s}: {v:.4f}')

print('\nXGBoost Metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k:>15s}: {v:.4f}')

## 3. Statistical Tests

In [ ]:
delong = delong_test(y_test, y_prob_nam, y_prob_xgb)
print(f"DeLong Test:")
print(f"  NAM AUC: {delong['auc_a']:.4f}")
print(f"  XGB AUC: {delong['auc_b']:.4f}")
print(f"  Diff: {delong['auc_diff']:.4f}")
print(f"  z-stat: {delong['z_statistic']:.4f}")
print(f"  p-value: {delong['p_value']:.4f}")

thresh_nam = nam_metrics['threshold']
thresh_xgb = xgb_metrics['threshold']
mcnemar = mcnemar_test(y_test, (y_prob_nam >= thresh_nam).astype(int), (y_prob_xgb >= thresh_xgb).astype(int))
print(f"\nMcNemar Test: chi2={mcnemar['chi2']:.4f}, p={mcnemar['p_value']:.4f}")

## 4. Bootstrap CIs

In [ ]:
nam_boot = bootstrap_metrics(y_test, y_prob_nam, n_bootstrap=1000)
xgb_boot = bootstrap_metrics(y_test, y_prob_xgb, n_bootstrap=1000)

print('NAM (95% CI):')
for k, v in nam_boot.items():
    print(f"  {k:>12s}: {v['mean']:.4f} ({v['lower']:.4f}, {v['upper']:.4f})")

print('\nXGBoost (95% CI):')
for k, v in xgb_boot.items():
    print(f"  {k:>12s}: {v['mean']:.4f} ({v['lower']:.4f}, {v['upper']:.4f})")

## 5. Conformal Prediction Comparison

In [ ]:
alpha_levels = config['conformal']['alpha_levels']

nam_wrapper = NAMSklearnWrapper(nam_model)
nam_wrapper.fit(X_cal, y_cal)
nam_conf = create_conformal_classifier(nam_wrapper, X_cal, y_cal)
nam_conf_res = predict_with_confidence(nam_conf, X_test, alpha_levels)

xgb_conf = create_conformal_classifier(xgb_model, X_cal, y_cal)
xgb_conf_res = predict_with_confidence(xgb_conf, X_test, alpha_levels)

nam_cov = conformal_coverage_analysis(y_test, nam_conf_res['y_sets'], alpha_levels)
xgb_cov = conformal_coverage_analysis(y_test, xgb_conf_res['y_sets'], alpha_levels)

## 6. Full Comparison Table

In [ ]:
table = create_comparison_table(
    nam_metrics, xgb_metrics,
    nam_boot, xgb_boot,
    nam_cov, xgb_cov,
)
table

## 7. Key Visualizations

In [ ]:
# ROC curves
fig = plot_roc_curves(y_test, y_prob_nam, y_prob_xgb, nam_metrics['auc_roc'], xgb_metrics['auc_roc'])
plt.show()

In [ ]:
# NAM vs SHAP comparison
shap_data = extract_shap_values(xgb_model, X_test, feature_names)
nam_imp = get_feature_importance_from_nam(nam_model, X_train)
top_3 = np.argsort(nam_imp)[::-1][:3].tolist()

fig = plot_nam_vs_shap_comparison(nam_model, shap_data, top_3, X_train, feature_names)
plt.show()

In [ ]:
# Feature importance comparison
shap_imp = np.abs(shap_data['shap_values']).mean(axis=0)
fig = plot_feature_importance_comparison(nam_imp, shap_imp, feature_names)
plt.show()